# VK AI Challenge: воспроизводимое финальное решение

Итоговый пайплайн использует ансамбль из 4 бустингов (`cat_all`, `cat_recent`, `lgb_neutral`, `lgb_diverse`), усреднение вероятностей по seed-ам `42, 2026, 777` и равные веса. Решающее правило — top-k с `k = 1.8 × 990 = 1782`, где число позитивов hidden test выводится из опубликованного `sample_f1 = 0.2347083926`. Подробное описание находится в [README.md](README.md) и [docs/HYPOTHESES_ROUND2.md](docs/HYPOTHESES_ROUND2.md).

In [1]:
from pathlib import Path
import subprocess
import sys

# Colab: install the pinned dependency set. Falls back to explicit pins when the
# notebook is run before the repository has been cloned.
requirements = Path("requirements.txt")
if requirements.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements)])
else:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "numpy==2.2.1", "pandas==2.2.3", "scikit-learn==1.6.0",
        "catboost==1.2.10", "lightgbm==4.7.0", "scipy==1.14.1", "joblib",
    ])


Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
import sys
from pathlib import Path
import subprocess

# The data files ship with the repository, so a fresh Colab runtime only needs a clone.
if not (Path("train.csv").exists() and Path("test.csv").exists()):
    clone_dir = Path("vk16")
    if not clone_dir.exists():
        subprocess.check_call(["git", "clone", "https://github.com/rnva822-alt/vk16.git", str(clone_dir)])
    os.chdir(clone_dir)
sys.path.insert(0, str(Path.cwd()))
print("Working directory:", Path.cwd())


Working directory: /home/ubuntu/repos/vk16


In [3]:
import solution

# Trains every component on the full training set (3 seeds each), blends them with equal
# weights and writes submission.csv with k = RATE_MULTIPLIER * TEST_POSITIVES positives.
solution.main()



Computing aggregated count/frequency/statistical features on combined train+test...
  content_owner_complaint_count: 23221 unique owners
  content_complaint_count:       47797 unique contents
  claim_type_count:              16 unique types
  claim_reason_count:            18 unique reasons
  Frequency encoding: 5 features (id_content_owner, claim_type, claim_reason, os, registered_phone_country)
  Statistical aggregations: 5 features (mean/std bot scores, mean likes by claim_type/reason)
FINAL: full train -> test.csv


    cat_all: iterations=362 (77.3s)


    cat_recent: iterations=382 (83.4s)


    lgb_neutral: iterations=432 (10.3s)


    lgb_diverse: iterations=1483 (31.3s)


{
  "rows_train": 48658,
  "rows_test": 7446,
  "seeds": [
    42,
    2026,
    777
  ],
  "cat_all_iterations": 362,
  "cat_all_seconds": 77.3,
  "cat_recent_iterations": 382,
  "cat_recent_seconds": 83.4,
  "lgb_neutral_iterations": 432,
  "lgb_neutral_seconds": 10.3,
  "lgb_diverse_iterations": 1483,
  "lgb_diverse_seconds": 31.3
}
submission.csv written with positives=1782, k=1782


In [4]:
import hashlib
import pandas as pd

submission = pd.read_csv("submission.csv")
sha256 = hashlib.sha256(Path("submission.csv").read_bytes()).hexdigest()
print(submission.head())
print("positive predictions:", int(submission["is_valid"].sum()))
print("positive share:", float(submission["is_valid"].mean()))
print("sha256:", sha256)

                                            claim_id  is_valid
0  486ac055356e93bb0a5fc90cb85a47e94e3a290364a775...         0
1  99c0083e402ccb55c0ed358ec6b5344c7785192c985ba4...         0
2  2b58bb8d601c0af249492aef72704568d249c5a2823760...         0
3  a0444524c12e0e16b6b541513f92f404c037031d3beec6...         0
4  94d88dc3314fe941148fae63ab4f7efd0c5748e31f7e82...         0
positive predictions: 1782
positive share: 0.23932312651087834
sha256: bd1fb0bb83474b8cd00b4af1f691d424aef3c03479da14f030c45bb45a2f2617
